Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [42]:
NO_RECORDS = 25

In [43]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests

In [44]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [45]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : "127.0.0.1"
}

In [46]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
LIMIT %s
"""

In [47]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()

        cur.execute(query_records, params = (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '8393ec5c-7723-4f4e-9d63-f27fcad32e36::doi:10.34894/VLN5PZ', 'title': 'Clinical relevance of verbal memory impairment in polydrug Ecstasy users: A pooled data analysis', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>doi:10.34894/VLN5PZ</identifier>\n    <datestamp>2025-07-05T00:00:18Z</datestamp>\n    <setSpec>MaastrichtUniversity</setSpec>\n    <setSpec>dataversenl</setSpec>\n  </header>\n  <metadata>\n    <resource xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n      <identifier identifierType="DOI">10.34894/VLN5PZ</identifier>\n      <creators>\n        <creator>\n          <creatorName nameType="Personal">Kuypers, K. P. C</creatorName>\n          <givenName>K. P. C</givenName>\n          <familyName>Kuype

In [48]:
record_titles[0]

'Clinical relevance of verbal memory impairment in polydrug Ecstasy users: A pooled data analysis'

In [49]:
record_raw_metadata[0]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>doi:10.34894/VLN5PZ</identifier>\n    <datestamp>2025-07-05T00:00:18Z</datestamp>\n    <setSpec>MaastrichtUniversity</setSpec>\n    <setSpec>dataversenl</setSpec>\n  </header>\n  <metadata>\n    <resource xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">\n      <identifier identifierType="DOI">10.34894/VLN5PZ</identifier>\n      <creators>\n        <creator>\n          <creatorName nameType="Personal">Kuypers, K. P. C</creatorName>\n          <givenName>K. P. C</givenName>\n          <familyName>Kuypers</familyName>\n          <affiliation>Maastricht University</affiliation>\n        </creator>\n        <creator>\n          <creatorName nameType="Personal">Theunissen, E. L</creatorName>\n   

In [50]:
def ET_from_string(string):
    return ET.ElementTree(ET.fromstring(string))

In [51]:
raw_xml = ET_from_string(record_raw_metadata[0])

In [52]:
raw_xml.getroot()

<Element {http://www.openarchives.org/OAI/2.0/}record at 0x1a25eed1280>

In [53]:
print(ET.tostring(raw_xml, pretty_print=True).decode())

<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">
  <header>
    <identifier>doi:10.34894/VLN5PZ</identifier>
    <datestamp>2025-07-05T00:00:18Z</datestamp>
    <setSpec>MaastrichtUniversity</setSpec>
    <setSpec>dataversenl</setSpec>
  </header>
  <metadata>
    <resource xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.1/metadata.xsd">
      <identifier identifierType="DOI">10.34894/VLN5PZ</identifier>
      <creators>
        <creator>
          <creatorName nameType="Personal">Kuypers, K. P. C</creatorName>
          <givenName>K. P. C</givenName>
          <familyName>Kuypers</familyName>
          <affiliation>Maastricht University</affiliation>
        </creator>
        <creator>
          <creatorName nameType="Personal">Theunissen, E. L</creatorName>
          <givenName>E.

In [54]:
# Finding a desired tag anywhere in tree, e.g. title
raw_xml.xpath("//*[local-name()='title']/text()")

['Clinical relevance of verbal memory impairment in polydrug Ecstasy users: A pooled data analysis']

In [55]:
element  = raw_xml.findall(".//{*}title")
print(element)



print(element[0].text)

[<Element {http://datacite.org/schema/kernel-4}title at 0x1a25d1ab700>]
Clinical relevance of verbal memory impairment in polydrug Ecstasy users: A pooled data analysis


In [56]:
element  = raw_xml.findall(".//{*}identifier[@identifierType='DOI']")
element[0].text

'10.34894/VLN5PZ'

In [57]:
# getting the DOI of the dataset
# Later, find related DOIs, articles...

raw_xml.xpath("//*[local-name()='identifier'][@identifierType='DOI']/text()")

['10.34894/VLN5PZ']